In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models
import os

DATASET_PATH = "dataset"
print(os.listdir("dataset"))
img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)

normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

epochs = 20

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)
model.save("pet_disease_model.keras")

['Dental Disease', 'Ear Mites', 'Eye Infection', 'Fungal Infection', 'Hot Spots', 'Mange', 'Ringworm', 'Scabies', 'Tick Infestation']
Found 890 files belonging to 9 classes.
Using 712 files for training.
Found 890 files belonging to 9 classes.
Using 178 files for validation.
Classes: ['Dental Disease', 'Ear Mites', 'Eye Infection', 'Fungal Infection', 'Hot Spots', 'Mange', 'Ringworm', 'Scabies', 'Tick Infestation']
Number of classes: 9
Epoch 1/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 14s 470ms/step - accuracy: 0.2556 - loss: 2.0853 - val_accuracy: 0.4101 - val_loss: 1.6456
Epoch 2/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 11s 458ms/step - accuracy: 0.5183 - loss: 1.3765 - val_accuracy: 0.4944 - val_loss: 1.4021
Epoch 3/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 12s 496ms/step - accuracy: 0.6517 - loss: 1.0436 - val_accuracy: 0.5225 - val_loss: 1.4280
Epoch 4/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 12s 510ms/step - accuracy: 0.6994 - loss: 0.8692 - val_accuracy: 0.5618 - val_loss: 1.2902
Epoch 5/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 13s 51